In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/sarah-ahmad-cv/Sarah Ahmad (CV).pdf


In [2]:
!pip install langchain langchain-community langchain-core transformers==4.52.4
from pypdf import PdfReader

INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 74.8 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 62.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 84.6 MB/s eta 0:00:00:00:01
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0rc2
    Uninstalling packaging-26.0rc2:
      Successfully uninstalled packaging-26.0rc2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.1
    Uninstalling tokenizers-0.22.1:
      Successfully uninstalled tokenizers-0.22.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.1
    Uninstalling transformers-4.57.1:
      Successfully uninstalled transformers

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain.output_parsers import StructuredOutputParser, ResponseSchema
from langchain.prompts import PromptTemplate
import torch
import re

model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

def generate_text(prompt, max_length=1000, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

2026-02-01 20:24:41.062397: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769977481.248956      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769977481.302028      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769977481.782085      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769977481.782118      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769977481.782121      55 computation_placer.cc:177] computation placer alr

...

model-00001-of-00005.safetensors:   0%|          | 0.00/4.87G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [4]:
fullname_schema = ResponseSchema(
    name = "FullName",
    description = "The Name of the candidate"
)
email_schema = ResponseSchema(
    name = "Email",
    description = "The Email address of the candidates"
)
education_schema = ResponseSchema(
    name = "Education",
    description = "List of degree, institution and year"
)
skills_schema = ResponseSchema(
    name = "Skills",
    description = "List of strings contains the skills’ names"
)
experience_schema = ResponseSchema(
    name = "Experience",
    description = "List of role, company and years"
)

In [5]:
response_schemas = [fullname_schema , email_schema , education_schema , skills_schema , experience_schema]
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

In [107]:
print(output_parser)
print(format_instructions)
print(type(output_parser))
print(type(format_instructions))


response_schemas=[ResponseSchema(name='FullName', description='The Name of the candidate', type='string'), ResponseSchema(name='Email', description='The Email address of the candidates', type='string'), ResponseSchema(name='Education', description='List of degree, institution and year', type='string'), ResponseSchema(name='Skills', description='List of strings contains the skills’ names', type='string'), ResponseSchema(name='Experience', description='List of role, company and years', type='string')]
The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"FullName": string  // The Name of the candidate
	"Email": string  // The Email address of the candidates
	"Education": string  // List of degree, institution and year
	"Skills": string  // List of strings contains the skills’ names
	"Experience": string  // List of role, company and years
}
```
<class 'langchain.output_parsers.structured.Struct

In [87]:
def extract_text_from_pdf(cv):
    reader = PdfReader(cv)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    return text

In [97]:
cv_extraction_template = """
You are a smart assistance that extracts the person’s:
- Fullname
- Email
- Education (degree, institution, year)
- skills
- Experience (role, company, year)
from the cv.
Respone only in JSON format as follows:
{format_instructions}
CV Text:
{cv_text}
"""

In [98]:
cv_path = "/kaggle/input/sarah-ahmad-cv/Sarah Ahmad (CV).pdf"
cv_text = extract_text_from_pdf(cv_path)

In [99]:
prompt_template = PromptTemplate(
    template = cv_extraction_template,
    input_variables = ['FullName', 'Email', 'Education', 'Skills', 'Experience']
)

In [100]:
prompt = prompt_template.format(
    cv_text = cv_text,
    format_instructions = format_instructions
)
response = generate_text(prompt, max_length = 1500)[0]    

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [102]:
def extract_json_block(text):
    pattern = r'```json\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL)
    return f"```json\n{matches[-1]}\n```"

In [103]:
json_text = extract_json_block(response)
output_data = output_parser.parse(json_text)
print(output_data)

{'FullName': 'Sarah Ahmad Saad', 'Email': 'sara.ahmed84258@gmail.com', 'Education': 'Faculty of Computer Science, Minya University | 2022 - 2026', 'Skills': ['Data Analysis', 'Data Visualization', 'LLMS', 'Generative AI', 'Deep Learning', 'Computer Vision', 'NLP', 'Azure AI', 'MLOps', 'RAG', 'LangChain', 'Python', 'C++', 'C', 'C#', 'OOP'], 'Experience': ['Machine Learning Trainee, Digital Egyptian Pioneers Initiative | Nov 2025 - Now', 'LLMS Intern, Tips Hindawi | Dec 2025 - Now', 'ICPC Trainee | Oct 2022 - Mar 2023']}


In [104]:
print("Full Name: ", output_data["FullName"])
print("Email: ", output_data["Email"])
print("Education: ", output_data["Education"])
print("Skills: ", output_data["Skills"])
print("Experience: ", output_data["Experience"])


Full Name:  Sarah Ahmad Saad
Email:  sara.ahmed84258@gmail.com
Education:  Faculty of Computer Science, Minya University | 2022 - 2026
Skills:  ['Data Analysis', 'Data Visualization', 'LLMS', 'Generative AI', 'Deep Learning', 'Computer Vision', 'NLP', 'Azure AI', 'MLOps', 'RAG', 'LangChain', 'Python', 'C++', 'C', 'C#', 'OOP']
Experience:  ['Machine Learning Trainee, Digital Egyptian Pioneers Initiative | Nov 2025 - Now', 'LLMS Intern, Tips Hindawi | Dec 2025 - Now', 'ICPC Trainee | Oct 2022 - Mar 2023']
